나이 예측 함수

In [ ]:
def predict_age(sentence: str):
    # 모델이 올라간 디바이스 확인
    device = next(model.parameters()).device

    # 토크나이징 + device 이동
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 예측
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()

    return predicted_class + 6  # 클래스 → 나이



In [ ]:
test_sentence = "아"
predicted_age = predict_age(test_sentence)
print(f"예측된 나이: {predicted_age}세")



예측된 나이: 9세


검증 데이터의 정답 라벨 분포 보기

In [ ]:
import numpy as np
unique, counts = np.unique(val_labels, return_counts=True)
print(dict(zip(unique, counts)))  # 라벨 0~6 분포 확인


{np.int64(0): np.int64(283), np.int64(1): np.int64(12336), np.int64(2): np.int64(13910), np.int64(3): np.int64(19678), np.int64(4): np.int64(13602), np.int64(5): np.int64(11191), np.int64(6): np.int64(9396)}


In [ ]:
preds = []
for s in val_texts[:100]:
    preds.append(predict_age(s))

import collections
print(collections.Counter(preds))


Counter({9: 100})


예측 테스트

✅ 전체 요약 흐름
📊 DataFrame에서 sentence/age 추출 (df['sentence'], df['age'])

🧠 age - 6 → 클래스 라벨로 변환 (총 7개 클래스: 6~12세)

🔤 KoBERT 토크나이저로 문장 인코딩

🔁 Trainer로 KoBERT 분류 모델 학습 (클래스 수 = 7)

🔍 새로운 문장을 입력하면 나이 예측